# Posicionamiento de precios

Este notebook responde exclusivamente la pregunta: **¿Dónde está Rappi vs. competencia en posicionamiento de precios: más caro, más barato o similar?**

## Limitación clave del dataset
- `Rappi` no tiene métricas directas de `Delivery Fee (MXN)` ni `Service Fee (%)` en el archivo.
- Por eso, el posicionamiento de Rappi se evalúa **de forma indirecta** con dos proxies:
  - `Restaurants Markdowns / GMV`: intensidad promocional o subsidio para sostener competitividad percibida.
  - `Gross Profit UE`: rentabilidad unitaria capturada por Rappi después de esa estrategia.
- `Uber Eats` y `DiDi Food` sí permiten comparación **directa** en fees.

La conclusión final debe leerse como **posicionamiento percibido / presión promocional**, no como una prueba del `precio base` de Rappi.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
RESOURCES = ROOT / "Resources"
FILEPATH = next(RESOURCES.glob("*.xlsx"))

metrics = pd.read_excel(FILEPATH, sheet_name="Input_metrics")
orders = pd.read_excel(FILEPATH, sheet_name="Orders")

LATEST_ROLL = "L0W_ROLL"
LATEST_ORDERS = "L0W"
ROLLING_WEEKS = [f"L{i}W_ROLL" for i in range(8, -1, -1)]
ORDERS_WEEKS = [f"L{i}W" for i in range(8, -1, -1)]

print(FILEPATH.name)
print("Input_metrics:", metrics.shape)
print("Orders:", orders.shape)

In [ ]:
metrics.groupby("COMPETITOR")["METRIC"].unique().apply(list)

## 1. Preparación de datasets analíticos

Se crean dos marcos:
- `fee_snapshot`: Uber y DiDi con fees directos observables.
- `rappi_snapshot`: Rappi con proxies de subsidio y rentabilidad.

In [ ]:
fee_metrics = ["Delivery Fee (MXN)", "Service Fee (%)"]
rappi_metrics = ["Restaurants Markdowns / GMV", "Gross Profit UE"]

fee_snapshot = (
    metrics.loc[
        metrics["COMPETITOR"].isin(["Uber Eats", "DiDi Food"])
        & metrics["METRIC"].isin(fee_metrics),
        ["COMPETITOR", "ZONE", "ZONE_TYPE", "METRIC", LATEST_ROLL],
    ]
    .pivot_table(
        index=["COMPETITOR", "ZONE", "ZONE_TYPE"],
        columns="METRIC",
        values=LATEST_ROLL,
    )
    .reset_index()
)

fee_snapshot = fee_snapshot.merge(
    orders.loc[
        orders["COMPETITOR"].isin(["Uber Eats", "DiDi Food"]),
        ["COMPETITOR", "ZONE", LATEST_ORDERS],
    ].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on=["COMPETITOR", "ZONE"],
    how="left",
)

for basket in [100, 150, 200, 250]:
    fee_snapshot[f"Effective Fee {basket}"] = (
        fee_snapshot["Delivery Fee (MXN)"]
        + basket * fee_snapshot["Service Fee (%)"]
    )

rappi_snapshot = (
    metrics.loc[
        (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(rappi_metrics),
        ["ZONE", "ZONE_TYPE", "METRIC", LATEST_ROLL],
    ]
    .pivot_table(index=["ZONE", "ZONE_TYPE"], columns="METRIC", values=LATEST_ROLL)
    .reset_index()
)

rappi_snapshot = rappi_snapshot.merge(
    orders.loc[
        orders["COMPETITOR"] == "Rappi",
        ["ZONE", LATEST_ORDERS],
    ].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on="ZONE",
    how="left",
)

fee_snapshot.head()

In [ ]:
rappi_snapshot.head()

## 2. Benchmark directo de fees: Uber Eats vs DiDi Food

Como no hay fees de Rappi, el benchmark directo de precio observable se limita a Uber y DiDi.

Para hacer comparable el costo al usuario se usa también `Effective Fee`, definido como:

`Delivery Fee + basket_value * Service Fee (%)`

El basket de referencia principal será **MXN 150**, suficientemente representativo para un pedido estándar.

In [ ]:
fee_summary = (
    fee_snapshot.groupby("COMPETITOR")
    .agg(
        zones=("ZONE", "nunique"),
        orders=("ORDERS", "sum"),
        delivery_fee_mean=("Delivery Fee (MXN)", "mean"),
        service_fee_mean=("Service Fee (%)", "mean"),
        effective_fee_150_mean=("Effective Fee 150", "mean"),
        effective_fee_200_mean=("Effective Fee 200", "mean"),
    )
    .reset_index()
)

weighted_fee_summary = pd.DataFrame([
    {
        "COMPETITOR": competitor,
        "delivery_fee_weighted": np.average(group["Delivery Fee (MXN)"], weights=group["ORDERS"]),
        "service_fee_weighted": np.average(group["Service Fee (%)"], weights=group["ORDERS"]),
        "effective_fee_150_weighted": np.average(group["Effective Fee 150"], weights=group["ORDERS"]),
        "effective_fee_200_weighted": np.average(group["Effective Fee 200"], weights=group["ORDERS"]),
    }
    for competitor, group in fee_snapshot.groupby("COMPETITOR")
])

fee_summary.merge(weighted_fee_summary, on="COMPETITOR")

In [ ]:
fig = px.bar(
    weighted_fee_summary.melt(
        id_vars="COMPETITOR",
        value_vars=["delivery_fee_weighted", "effective_fee_150_weighted", "effective_fee_200_weighted"],
        var_name="metric",
        value_name="value",
    ),
    x="metric",
    y="value",
    color="COMPETITOR",
    barmode="group",
    text_auto=".2f",
    title="Costo observable ponderado por órdenes: Uber Eats vs DiDi Food",
    labels={"metric": "Métrica", "value": "MXN"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
zone_fee_comp = (
    fee_snapshot.pivot_table(
        index=["ZONE", "ZONE_TYPE"],
        columns="COMPETITOR",
        values=["Delivery Fee (MXN)", "Service Fee (%)", "Effective Fee 150"],
    )
)
zone_fee_comp.columns = [f"{metric}|{competitor}" for metric, competitor in zone_fee_comp.columns]
zone_fee_comp = zone_fee_comp.reset_index()
zone_fee_comp["cheapest_effective_fee_150"] = np.where(
    zone_fee_comp["Effective Fee 150|Uber Eats"] < zone_fee_comp["Effective Fee 150|DiDi Food"],
    "Uber Eats",
    "DiDi Food",
)
zone_fee_comp["gap_uber_minus_didi"] = (
    zone_fee_comp["Effective Fee 150|Uber Eats"] - zone_fee_comp["Effective Fee 150|DiDi Food"]
)
zone_fee_comp.sort_values("gap_uber_minus_didi")

In [ ]:
heatmap_df = zone_fee_comp.sort_values("gap_uber_minus_didi").copy()

fig = px.bar(
    heatmap_df,
    x="gap_uber_minus_didi",
    y="ZONE",
    color="cheapest_effective_fee_150",
    orientation="h",
    title="Gap de costo observable por zona (Uber Eats - DiDi Food) para basket de MXN 150",
    labels={"gap_uber_minus_didi": "Gap MXN", "ZONE": "Zona"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white")
fig.show()

## 3. Rappi: posicionamiento indirecto vía subsidio y rentabilidad

La lógica es:
- `Restaurants Markdowns / GMV` alto sugiere mayor necesidad de subsidio/promoción.
- `Gross Profit UE` alto sugiere mejor captura de margen por orden.
- Si Rappi necesita más markdown y aun así captura poco margen, hay señal de presión competitiva en precio.
- Si Rappi subsidia menos y mantiene mayor margen, su posición de precio/rentabilidad parece más sólida.

In [ ]:
rappi_summary = pd.DataFrame([
    {
        "markdown_mean": rappi_snapshot["Restaurants Markdowns / GMV"].mean(),
        "markdown_weighted": np.average(rappi_snapshot["Restaurants Markdowns / GMV"], weights=rappi_snapshot["ORDERS"]),
        "gross_profit_mean": rappi_snapshot["Gross Profit UE"].mean(),
        "gross_profit_weighted": np.average(rappi_snapshot["Gross Profit UE"], weights=rappi_snapshot["ORDERS"]),
        "corr_markdown_vs_gp": rappi_snapshot[["Restaurants Markdowns / GMV", "Gross Profit UE"]].corr().iloc[0, 1],
    }
])
rappi_summary

In [ ]:
rappi_zone_type = pd.DataFrame([
    {
        "ZONE_TYPE": zone_type,
        "zones": len(group),
        "orders_total": group["ORDERS"].sum(),
        "markdown_mean": group["Restaurants Markdowns / GMV"].mean(),
        "markdown_weighted": np.average(group["Restaurants Markdowns / GMV"], weights=group["ORDERS"]),
        "gross_profit_mean": group["Gross Profit UE"].mean(),
        "gross_profit_weighted": np.average(group["Gross Profit UE"], weights=group["ORDERS"]),
    }
    for zone_type, group in rappi_snapshot.groupby("ZONE_TYPE")
])
rappi_zone_type

In [ ]:
fig = px.scatter(
    rappi_snapshot,
    x="Restaurants Markdowns / GMV",
    y="Gross Profit UE",
    size="ORDERS",
    color="ZONE_TYPE",
    hover_name="ZONE",
    title="Rappi: trade-off entre intensidad promocional y rentabilidad unitaria",
    labels={
        "Restaurants Markdowns / GMV": "Markdowns / GMV",
        "Gross Profit UE": "Gross Profit UE",
    },
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
rappi_priority_zones = rappi_snapshot.sort_values(
    ["Restaurants Markdowns / GMV", "Gross Profit UE"],
    ascending=[False, True],
).copy()
rappi_priority_zones[[
    "ZONE",
    "ZONE_TYPE",
    "Restaurants Markdowns / GMV",
    "Gross Profit UE",
    "ORDERS",
]]

## 4. Tendencia reciente de presión promocional y fees observables

Esto no mezcla métricas incompatibles para comparar niveles absolutos, pero sí ayuda a leer si:
- los fees observables de Uber/DiDi se han estabilizado o encarecido,
- Rappi ha aumentado o reducido su presión promocional reciente.

In [ ]:
trend_rows = []

for roll_week, orders_week in zip(ROLLING_WEEKS, ORDERS_WEEKS):
    rappi_week = (
        metrics.loc[
            (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(rappi_metrics),
            ["ZONE", "METRIC", roll_week],
        ]
        .pivot_table(index="ZONE", columns="METRIC", values=roll_week)
        .reset_index()
    )
    rappi_week = rappi_week.merge(
        orders.loc[orders["COMPETITOR"] == "Rappi", ["ZONE", orders_week]].rename(columns={orders_week: "ORDERS"}),
        on="ZONE",
        how="left",
    )
    trend_rows.append(
        {
            "week": roll_week,
            "series": "Rappi Markdown / GMV",
            "value": np.average(rappi_week["Restaurants Markdowns / GMV"], weights=rappi_week["ORDERS"]),
        }
    )
    trend_rows.append(
        {
            "week": roll_week,
            "series": "Rappi Gross Profit UE",
            "value": np.average(rappi_week["Gross Profit UE"], weights=rappi_week["ORDERS"]),
        }
    )

    for competitor in ["Uber Eats", "DiDi Food"]:
        comp_week = (
            metrics.loc[
                (metrics["COMPETITOR"] == competitor) & metrics["METRIC"].isin(fee_metrics),
                ["ZONE", "METRIC", roll_week],
            ]
            .pivot_table(index="ZONE", columns="METRIC", values=roll_week)
            .reset_index()
        )
        comp_week = comp_week.merge(
            orders.loc[orders["COMPETITOR"] == competitor, ["ZONE", orders_week]].rename(columns={orders_week: "ORDERS"}),
            on="ZONE",
            how="left",
        )
        comp_week["Effective Fee 150"] = comp_week["Delivery Fee (MXN)"] + 150 * comp_week["Service Fee (%)"]
        trend_rows.append(
            {
                "week": roll_week,
                "series": f"{competitor} Effective Fee @150",
                "value": np.average(comp_week["Effective Fee 150"], weights=comp_week["ORDERS"]),
            }
        )

trend_df = pd.DataFrame(trend_rows)
trend_df

In [ ]:
fig = px.line(
    trend_df[trend_df["series"].isin(["Uber Eats Effective Fee @150", "DiDi Food Effective Fee @150"])],
    x="week",
    y="value",
    color="series",
    markers=True,
    title="Tendencia reciente del costo observable ponderado por órdenes",
    labels={"value": "MXN", "week": "Semana rolling"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.line(
    trend_df[trend_df["series"].isin(["Rappi Markdown / GMV", "Rappi Gross Profit UE"])],
    x="week",
    y="value",
    color="series",
    markers=True,
    title="Tendencia reciente de presión promocional y rentabilidad unitaria en Rappi",
    labels={"value": "Valor", "week": "Semana rolling"},
)
fig.update_layout(template="plotly_white")
fig.show()

## 5. Hallazgos ejecutivos

### Qué sí se puede afirmar con los datos
- `DiDi Food` aparece como el jugador con menor costo observable al usuario en promedio ponderado por órdenes.
- `Uber Eats` es más caro que DiDi tanto en `Delivery Fee` como en `Effective Fee` para baskets de referencia.
- La ventaja de DiDi no es homogénea: hay zonas donde Uber sigue siendo más barato en fee efectivo.
- En Rappi, los `markdowns` son materialmente más altos en zonas `Non Wealthy`, mientras que el `Gross Profit UE` es mucho menor e incluso negativo en varias de ellas.
- La relación negativa entre `Restaurants Markdowns / GMV` y `Gross Profit UE` sugiere que cuando Rappi necesita subsidiar más, erosiona significativamente su rentabilidad unitaria.

### Qué no se puede afirmar de forma directa
- No se puede clasificar de manera concluyente a `Rappi` como `más caro`, `más barato` o `similar` en `precio base`, porque el dataset no incluye sus `Delivery Fee` ni `Service Fee`.

### Lectura ejecutiva sugerida para la presentación
- **DiDi lidera en precio observable** frente a Uber Eats en el promedio de mercado analizado.
- **Rappi parece sostener competitividad percibida vía subsidio**, especialmente en zonas `Non Wealthy`, donde los descuentos relativos son más altos y la rentabilidad por orden se deteriora.
- **Hipótesis de negocio**: Rappi probablemente enfrenta mayor presión competitiva en precio en zonas sensibles al costo, pero requiere data directa de fees para confirmar si esa presión proviene de un `precio base` menos competitivo o de una estrategia promocional agresiva.

## 6. Ventaja/desventaja operacional: comparación de tiempos de entrega

### Limitación metodológica
- `Rappi` no tiene la métrica `Avg Delivery Time (mins)` en el dataset.
- Por eso, la comparación operativa se divide en dos capas:
  - `Uber Eats` vs `DiDi Food`: comparación **directa** de tiempos de entrega.
  - `Rappi`: evaluación **indirecta** con proxies operativos como `Perfect Orders`, `Turbo Adoption`, `Lead Penetration` y tasas de conversión (`SST > SS CVR`).

Este marco permite inferir fortaleza o tensión operacional, pero **no equivale** a una medición directa de tiempo de entrega para Rappi.

In [ ]:
time_snapshot = (
    metrics.loc[
        metrics["COMPETITOR"].isin(["Uber Eats", "DiDi Food"])
        & (metrics["METRIC"] == "Avg Delivery Time (mins)"),
        ["COMPETITOR", "ZONE", "ZONE_TYPE", LATEST_ROLL],
    ]
    .rename(columns={LATEST_ROLL: "delivery_time_mins"})
)

time_snapshot = time_snapshot.merge(
    orders.loc[
        orders["COMPETITOR"].isin(["Uber Eats", "DiDi Food"]),
        ["COMPETITOR", "ZONE", LATEST_ORDERS],
    ].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on=["COMPETITOR", "ZONE"],
    how="left",
)

time_summary = pd.DataFrame([
    {
        "COMPETITOR": competitor,
        "zones": group["ZONE"].nunique(),
        "orders": group["ORDERS"].sum(),
        "delivery_time_mean": group["delivery_time_mins"].mean(),
        "delivery_time_weighted": np.average(group["delivery_time_mins"], weights=group["ORDERS"]),
    }
    for competitor, group in time_snapshot.groupby("COMPETITOR")
])

time_summary

In [ ]:
zone_time_comp = (
    time_snapshot.pivot_table(
        index=["ZONE", "ZONE_TYPE"],
        columns="COMPETITOR",
        values="delivery_time_mins",
    )
    .dropna()
    .reset_index()
)
zone_time_comp["faster_competitor"] = np.where(
    zone_time_comp["Uber Eats"] < zone_time_comp["DiDi Food"],
    "Uber Eats",
    "DiDi Food",
)
zone_time_comp["gap_uber_minus_didi"] = zone_time_comp["Uber Eats"] - zone_time_comp["DiDi Food"]
zone_time_comp.sort_values("gap_uber_minus_didi")

In [ ]:
fig = px.bar(
    zone_time_comp.sort_values("gap_uber_minus_didi"),
    x="gap_uber_minus_didi",
    y="ZONE",
    color="faster_competitor",
    orientation="h",
    title="Gap de tiempo de entrega por zona (Uber Eats - DiDi Food)",
    labels={"gap_uber_minus_didi": "Minutos", "ZONE": "Zona"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
time_zone_type = pd.DataFrame([
    {
        "COMPETITOR": competitor,
        "ZONE_TYPE": zone_type,
        "delivery_time_mean": group["delivery_time_mins"].mean(),
        "delivery_time_weighted": np.average(group["delivery_time_mins"], weights=group["ORDERS"]),
    }
    for (competitor, zone_type), group in time_snapshot.groupby(["COMPETITOR", "ZONE_TYPE"])
])

fig = px.bar(
    time_zone_type,
    x="ZONE_TYPE",
    y="delivery_time_weighted",
    color="COMPETITOR",
    barmode="group",
    text_auto=".1f",
    title="Tiempo de entrega ponderado por órdenes por tipo de zona",
    labels={"delivery_time_weighted": "Minutos ponderados", "ZONE_TYPE": "Tipo de zona"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
rappi_operational_metrics = [
    "Perfect Orders",
    "Restaurants SST > SS CVR",
    "Retail SST > SS CVR",
    "Turbo Adoption",
    "Lead Penetration",
    "Non-Pro PTC > OP",
]

rappi_ops = (
    metrics.loc[
        (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(rappi_operational_metrics),
        ["ZONE", "ZONE_TYPE", "METRIC", LATEST_ROLL],
    ]
    .pivot_table(index=["ZONE", "ZONE_TYPE"], columns="METRIC", values=LATEST_ROLL)
    .reset_index()
)

rappi_ops = rappi_ops.merge(
    orders.loc[orders["COMPETITOR"] == "Rappi", ["ZONE", LATEST_ORDERS]].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on="ZONE",
    how="left",
)

for column in rappi_operational_metrics:
    rappi_ops[f"{column}_z"] = (
        rappi_ops[column] - rappi_ops[column].mean()
    ) / rappi_ops[column].std(ddof=0)

rappi_ops["operational_health_score"] = rappi_ops[[f"{column}_z" for column in rappi_operational_metrics]].mean(axis=1)

rappi_ops[["ZONE", "ZONE_TYPE", "ORDERS", "operational_health_score"] + rappi_operational_metrics].sort_values("operational_health_score")

In [ ]:
rappi_ops_zone_type = pd.DataFrame([
    {
        "ZONE_TYPE": zone_type,
        "zones": len(group),
        "orders_total": group["ORDERS"].sum(),
        "perfect_orders_mean": group["Perfect Orders"].mean(),
        "turbo_adoption_mean": group["Turbo Adoption"].mean(),
        "restaurants_cvr_mean": group["Restaurants SST > SS CVR"].mean(),
        "retail_cvr_mean": group["Retail SST > SS CVR"].mean(),
        "lead_penetration_mean": group["Lead Penetration"].mean(),
        "health_score_mean": group["operational_health_score"].mean(),
        "health_score_weighted": np.average(group["operational_health_score"], weights=group["ORDERS"]),
    }
    for zone_type, group in rappi_ops.groupby("ZONE_TYPE")
])

rappi_ops_zone_type

In [ ]:
fig = px.scatter(
    rappi_ops,
    x="Turbo Adoption",
    y="Perfect Orders",
    size="ORDERS",
    color="ZONE_TYPE",
    hover_name="ZONE",
    title="Rappi: mapa indirecto de salud operacional",
    labels={"Turbo Adoption": "Turbo Adoption", "Perfect Orders": "Perfect Orders"},
)
fig.update_layout(template="plotly_white")
fig.show()

## 7. Hallazgos ejecutivos: ventaja/desventaja operacional

### Qué sí se puede afirmar con los datos
- `Uber Eats` tiene una **ventaja operacional directa** frente a `DiDi Food` en tiempos de entrega: su promedio ponderado es menor y gana en 10 de 15 zonas comparables.
- La ventaja de Uber es especialmente marcada en varias zonas `Wealthy` y en algunos bolsillos `Non-Wealthy` como `Milpa Alta` e `Iztapalapa`.
- `DiDi Food` solo supera a Uber en un grupo más reducido de zonas, con fortalezas puntuales como `Coyoacán`, `Xochimilco` y `Narvarte`.
- En Rappi, la evidencia indirecta sugiere una operación más robusta en zonas `Wealthy`: mayor `Turbo Adoption`, mayor `Lead Penetration` y un `operational_health_score` claramente superior.
- En zonas `Non Wealthy`, Rappi muestra señales de mayor tensión operacional relativa: menor score agregado, menor adopción de soluciones rápidas y desempeño más débil en varios indicadores de calidad/conversión.

### Qué no se puede afirmar de forma directa
- No se puede concluir con precisión si `Rappi` entrega más rápido o más lento que `Uber Eats` o `DiDi Food`, porque el dataset no contiene su métrica de `Avg Delivery Time (mins)`.

### Lectura ejecutiva sugerida para la presentación
- **Uber Eats lidera operativamente** en el benchmark observable de tiempos de entrega.
- **DiDi Food queda rezagado** en velocidad promedio, aunque conserva ventajas localizadas en ciertas zonas.
- **Rappi probablemente presenta una operación heterogénea**: más sólida en zonas premium y más tensionada en zonas sensibles a cobertura/costo, donde sus proxies operativos se deterioran.
- **Implicación de negocio**: si Rappi quiere cerrar la brecha operacional percibida, el foco debería estar en zonas `Non Wealthy`, donde los indicadores indirectos sugieren más fricción operativa.

## 8. Estrategia promocional: qué tipo de descuentos usa cada competidor

### Limitación metodológica
- El dataset **no incluye una métrica directa de promociones activas** para `Uber Eats` ni `DiDi Food`.
- `Rappi` sí tiene una señal promocional explícita: `Restaurants Markdowns / GMV`.
- Por eso, la lectura promocional se separa en dos capas:
  - `Rappi`: análisis directo de intensidad de subsidio/promoción.
  - `Uber Eats` y `DiDi Food`: inferencia indirecta a partir de `Delivery Fee`, `Service Fee` y la estabilidad/variabilidad del fee efectivo.

Este enfoque permite identificar **cómo compite cada jugador por precio percibido**, aunque no distingue el detalle exacto de cada cupón o mecánica promocional.

In [ ]:
rappi_promo_trend_rows = []

for roll_week, orders_week in zip(ROLLING_WEEKS, ORDERS_WEEKS):
    rappi_week = (
        metrics.loc[
            (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(["Restaurants Markdowns / GMV", "Gross Profit UE"]),
            ["ZONE", "ZONE_TYPE", "METRIC", roll_week],
        ]
        .pivot_table(index=["ZONE", "ZONE_TYPE"], columns="METRIC", values=roll_week)
        .reset_index()
    )
    rappi_week = rappi_week.merge(
        orders.loc[orders["COMPETITOR"] == "Rappi", ["ZONE", orders_week]].rename(columns={orders_week: "ORDERS"}),
        on="ZONE",
        how="left",
    )
    rappi_promo_trend_rows.append(
        {
            "week": roll_week,
            "markdown_weighted": np.average(rappi_week["Restaurants Markdowns / GMV"], weights=rappi_week["ORDERS"]),
            "gross_profit_weighted": np.average(rappi_week["Gross Profit UE"], weights=rappi_week["ORDERS"]),
        }
    )

rappi_promo_trend = pd.DataFrame(rappi_promo_trend_rows)
rappi_promo_trend

In [ ]:
rappi_promo_latest = (
    metrics.loc[
        (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(["Restaurants Markdowns / GMV", "Gross Profit UE"]),
        ["ZONE", "ZONE_TYPE", "METRIC", LATEST_ROLL],
    ]
    .pivot_table(index=["ZONE", "ZONE_TYPE"], columns="METRIC", values=LATEST_ROLL)
    .reset_index()
)

rappi_promo_latest = rappi_promo_latest.merge(
    orders.loc[orders["COMPETITOR"] == "Rappi", ["ZONE", LATEST_ORDERS]].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on="ZONE",
    how="left",
)

rappi_promo_latest.sort_values("Restaurants Markdowns / GMV", ascending=False)

In [ ]:
fig = px.scatter(
    rappi_promo_latest,
    x="Restaurants Markdowns / GMV",
    y="Gross Profit UE",
    size="ORDERS",
    color="ZONE_TYPE",
    hover_name="ZONE",
    title="Rappi: intensidad promocional vs rentabilidad unitaria",
    labels={
        "Restaurants Markdowns / GMV": "Markdowns / GMV",
        "Gross Profit UE": "Gross Profit UE",
    },
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fee_strategy_trend_rows = []

for roll_week, orders_week in zip(ROLLING_WEEKS, ORDERS_WEEKS):
    snap = (
        metrics.loc[
            metrics["COMPETITOR"].isin(["Uber Eats", "DiDi Food"])
            & metrics["METRIC"].isin(["Delivery Fee (MXN)", "Service Fee (%)"]),
            ["COMPETITOR", "ZONE", "METRIC", roll_week],
        ]
        .pivot_table(index=["COMPETITOR", "ZONE"], columns="METRIC", values=roll_week)
        .reset_index()
    )
    snap = snap.merge(
        orders.loc[
            orders["COMPETITOR"].isin(["Uber Eats", "DiDi Food"]),
            ["COMPETITOR", "ZONE", orders_week],
        ].rename(columns={orders_week: "ORDERS"}),
        on=["COMPETITOR", "ZONE"],
        how="left",
    )
    snap["effective_fee_150"] = snap["Delivery Fee (MXN)"] + 150 * snap["Service Fee (%)"]

    for competitor, group in snap.groupby("COMPETITOR"):
        fee_strategy_trend_rows.append(
            {
                "week": roll_week,
                "COMPETITOR": competitor,
                "delivery_fee_weighted": np.average(group["Delivery Fee (MXN)"], weights=group["ORDERS"]),
                "service_fee_weighted": np.average(group["Service Fee (%)"], weights=group["ORDERS"]),
                "effective_fee_150_weighted": np.average(group["effective_fee_150"], weights=group["ORDERS"]),
            }
        )

fee_strategy_trend = pd.DataFrame(fee_strategy_trend_rows)
fee_strategy_trend

In [ ]:
fee_strategy_summary = fee_strategy_trend.groupby("COMPETITOR").agg(
    effective_fee_150_mean=("effective_fee_150_weighted", "mean"),
    effective_fee_150_std=("effective_fee_150_weighted", "std"),
    effective_fee_150_min=("effective_fee_150_weighted", "min"),
    effective_fee_150_max=("effective_fee_150_weighted", "max"),
).reset_index()

fee_strategy_summary

In [ ]:
fig = px.line(
    fee_strategy_trend,
    x="week",
    y="effective_fee_150_weighted",
    color="COMPETITOR",
    markers=True,
    title="Tendencia del fee efectivo ponderado por órdenes (basket MXN 150)",
    labels={"effective_fee_150_weighted": "MXN", "week": "Semana rolling"},
)
fig.update_layout(template="plotly_white")
fig.show()

## 9. Hallazgos ejecutivos: estrategia promocional

### Qué sí se puede afirmar con los datos
- `Rappi` es el único competidor con evidencia explícita de subsidio promocional en el dataset a través de `Restaurants Markdowns / GMV`.
- El nivel de `markdowns` de Rappi es estructuralmente alto y relativamente estable en el tiempo, lo que sugiere una estrategia promocional recurrente, no táctica o aislada.
- La intensidad promocional de Rappi es más alta en zonas `Non Wealthy`, donde además el `Gross Profit UE` cae con fuerza e incluso es negativo en varias zonas. Eso indica que Rappi usa descuentos para sostener competitividad en zonas más sensibles a precio.
- `DiDi Food` parece competir con una estrategia más orientada a **precio bajo estructural**: mantiene el `effective fee` más bajo en promedio frente a `Uber Eats`.
- `Uber Eats` parece menos agresivo en precio final observable y más consistente en su arquitectura de fees; su `effective fee` es más alto y menos volátil que el de DiDi.

### Qué no se puede afirmar de forma directa
- No se puede identificar el detalle exacto de las promociones de `Uber Eats` o `DiDi Food` porque el archivo no trae descuentos activos, cupones o markdowns equivalentes.
- Para esos dos jugadores, la lectura promocional es una inferencia basada en su estructura de fees y precio final observable, no una observación directa de campañas.

### Lectura ejecutiva sugerida para la presentación
- **Rappi compite vía subsidio directo**, especialmente en zonas `Non Wealthy`, sacrificando margen para sostener competitividad percibida.
- **DiDi Food compite vía precio bajo estructural**, más que por señales visibles de subsidio en el dataset.
- **Uber Eats parece seguir una estrategia menos promocional y más premium/estable**, aceptando un fee efectivo mayor a cambio de no comprimir tanto el ingreso observable.
- **Implicación de negocio**: el mapa competitivo sugiere tres lógicas distintas de promoción: `Rappi` compra competitividad con descuentos, `DiDi` compite con precio de entrada bajo y `Uber Eats` protege mejor su monetización observable.

## 10. Variabilidad geográfica: ¿la competitividad de Rappi varía por zona?

### Enfoque metodológico
- Como el dataset no tiene precio final ni tiempo de entrega directo para `Rappi`, la competitividad geográfica se mide con un **score compuesto indirecto**.
- El score combina dos dimensiones:
  - `pricing_competitiveness_score`: mejor cuando Rappi necesita menos `Restaurants Markdowns / GMV` y captura mayor `Gross Profit UE`.
  - `operational_competitiveness_score`: mejor cuando Rappi muestra mejores proxies operativos (`Perfect Orders`, `Turbo Adoption`, `Lead Penetration`, `Non-Pro PTC > OP` y CVRs).
- El `overall_competitiveness_score` promedia ambas dimensiones para identificar dónde Rappi parece más fuerte o más frágil competitivamente.

In [ ]:
rappi_geo_metrics = [
    "Restaurants Markdowns / GMV",
    "Gross Profit UE",
    "Perfect Orders",
    "Restaurants SST > SS CVR",
    "Retail SST > SS CVR",
    "Turbo Adoption",
    "Lead Penetration",
    "Non-Pro PTC > OP",
]

rappi_geo = (
    metrics.loc[
        (metrics["COMPETITOR"] == "Rappi") & metrics["METRIC"].isin(rappi_geo_metrics),
        ["ZONE", "ZONE_TYPE", "METRIC", LATEST_ROLL],
    ]
    .pivot_table(index=["ZONE", "ZONE_TYPE"], columns="METRIC", values=LATEST_ROLL)
    .reset_index()
)

rappi_geo = rappi_geo.merge(
    orders.loc[orders["COMPETITOR"] == "Rappi", ["ZONE", LATEST_ORDERS]].rename(columns={LATEST_ORDERS: "ORDERS"}),
    on="ZONE",
    how="left",
)

benefit_metrics = [
    "Gross Profit UE",
    "Perfect Orders",
    "Restaurants SST > SS CVR",
    "Retail SST > SS CVR",
    "Turbo Adoption",
    "Lead Penetration",
    "Non-Pro PTC > OP",
]
cost_metrics = ["Restaurants Markdowns / GMV"]

for metric in benefit_metrics:
    rappi_geo[f"{metric}_z"] = (
        rappi_geo[metric] - rappi_geo[metric].mean()
    ) / rappi_geo[metric].std(ddof=0)

for metric in cost_metrics:
    rappi_geo[f"{metric}_z"] = -1 * (
        (rappi_geo[metric] - rappi_geo[metric].mean())
        / rappi_geo[metric].std(ddof=0)
    )

rappi_geo["pricing_competitiveness_score"] = rappi_geo[[
    "Restaurants Markdowns / GMV_z",
    "Gross Profit UE_z",
]].mean(axis=1)

rappi_geo["operational_competitiveness_score"] = rappi_geo[[
    f"{metric}_z" for metric in [
        "Perfect Orders",
        "Restaurants SST > SS CVR",
        "Retail SST > SS CVR",
        "Turbo Adoption",
        "Lead Penetration",
        "Non-Pro PTC > OP",
    ]
]].mean(axis=1)

rappi_geo["overall_competitiveness_score"] = rappi_geo[[
    "pricing_competitiveness_score",
    "operational_competitiveness_score",
]].mean(axis=1)

rappi_geo["competitive_quadrant"] = [
    (
        f"{'High' if pricing >= 0 else 'Low'} price proxy / "
        f"{'High' if ops >= 0 else 'Low'} ops"
    )
    for pricing, ops in zip(
        rappi_geo["pricing_competitiveness_score"],
        rappi_geo["operational_competitiveness_score"],
    )
]

rappi_geo[[
    "ZONE",
    "ZONE_TYPE",
    "ORDERS",
    "pricing_competitiveness_score",
    "operational_competitiveness_score",
    "overall_competitiveness_score",
    "competitive_quadrant",
]].sort_values("overall_competitiveness_score")

In [ ]:
rappi_geo_zone_type = pd.DataFrame([
    {
        "ZONE_TYPE": zone_type,
        "zones": len(group),
        "orders_total": group["ORDERS"].sum(),
        "pricing_score_mean": group["pricing_competitiveness_score"].mean(),
        "ops_score_mean": group["operational_competitiveness_score"].mean(),
        "overall_score_mean": group["overall_competitiveness_score"].mean(),
        "overall_score_weighted": np.average(group["overall_competitiveness_score"], weights=group["ORDERS"]),
        "markdown_mean": group["Restaurants Markdowns / GMV"].mean(),
        "gross_profit_mean": group["Gross Profit UE"].mean(),
    }
    for zone_type, group in rappi_geo.groupby("ZONE_TYPE")
])

rappi_geo_zone_type

In [ ]:
fig = px.scatter(
    rappi_geo,
    x="pricing_competitiveness_score",
    y="operational_competitiveness_score",
    size="ORDERS",
    color="ZONE_TYPE",
    hover_name="ZONE",
    title="Rappi: mapa de competitividad por zona",
    labels={
        "pricing_competitiveness_score": "Score de competitividad de precio (proxy)",
        "operational_competitiveness_score": "Score de competitividad operacional",
    },
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.bar(
    rappi_geo.sort_values("overall_competitiveness_score"),
    x="overall_competitiveness_score",
    y="ZONE",
    color="ZONE_TYPE",
    orientation="h",
    title="Ranking de competitividad total de Rappi por zona",
    labels={"overall_competitiveness_score": "Score total", "ZONE": "Zona"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
rappi_geo[[
    "ZONE",
    "ZONE_TYPE",
    "ORDERS",
    "Restaurants Markdowns / GMV",
    "Gross Profit UE",
    "overall_competitiveness_score",
    "competitive_quadrant",
]].sort_values("ORDERS", ascending=False).head(10)

## 11. Hallazgos ejecutivos: variabilidad geográfica

### Respuesta corta
- **Sí, la competitividad de Rappi varía fuertemente por zona.** La brecha más clara está entre zonas `Wealthy` y `Non Wealthy`.

### Qué muestran los datos
- Las zonas `Wealthy` concentran los mejores resultados de Rappi: menor necesidad de `markdowns`, mayor `Gross Profit UE` y mejores señales operativas agregadas.
- Las zonas `Non Wealthy` muestran el patrón opuesto: mayor subsidio, menor rentabilidad y señales operativas más débiles en promedio.
- El score compuesto separa casi perfectamente dos bloques: varias zonas `Wealthy` quedan en el cuadrante `High price proxy / High ops`, mientras que muchas `Non Wealthy` caen en `Low price proxy / Low ops`.
- Las zonas más fuertes para Rappi incluyen `Roma-Polanco`, `Del Valle-Narvarte`, `Santafe` e `Interlomas`.
- Las zonas más frágiles incluyen `CDMX_Cuautitlan_Melchor_Ocampo`, `Las Americas`, `Coacalco` y `CDMX_Agricola_Benito_Juarez`.
- La variabilidad no es solo académica: las dos zonas con mayor volumen (`Del Valle-Narvarte` y `Roma-Polanco`) también están entre las más competitivas, mientras que varios bolsillos `Non Wealthy` combinan menor competitividad con menor margen.

### Lectura ejecutiva sugerida para la presentación
- **Rappi no tiene una posición competitiva homogénea en CDMX.** Su propuesta es claramente más fuerte en corredores premium y más frágil en zonas sensibles a precio/cobertura.
- **En zonas premium, Rappi parece capturar una ventaja estructural**, con mejor monetización y mejor salud operativa.
- **En zonas no premium, Rappi parece comprar competitividad con más subsidio sin lograr la misma calidad de resultados**, lo que sugiere un modelo menos eficiente geográficamente.
- **Implicación de negocio**: la estrategia no debería ser uniforme por ciudad. Rappi necesita una gestión mucho más diferenciada por clúster geográfico, con foco de recuperación en zonas `Non Wealthy` y defensa/monetización en zonas `Wealthy`.